In [1]:
# 0. constants and raster paths

# NOTE:
# VERIFY that base dem (here, canwell_domainlong_elev2010_5m.tif) nodata is NOT NaN
# because NaN != NaN inherently, would throw false valid cells

CELL_SIZE = 5.0 # m, from elev raster
MIN_ELEV  = 805.0 # m, SET for plotting later

# combined slope+basin DEM domains = routing DEM
dem_path    = "../data/rasters/canwell_domainlong_elev2010_5m.tif"

# slope-domain-only feature rasters

aspect_path = "../data/rasters/canwell_domainlong_aspect2010_5m.tif"
eucdist_path = "../data/rasters/canwell_domainlong_dist_basin2010_5m.tif"
ndvi_path   = "../data/rasters/canwell_domainlong_ndvi2010_5m.tif"
lst_path    = "../data/rasters/canwell_domainlong_lst2010_5m.tif"
pddem_path   = "../data/rasters/canwell_domainlong_ddem_2010_2000_5m.tif" # feature
roughness_path = "../data/rasters/canwell_domainlong_roughness2010_5m.tif"
slope_path  = "../data/rasters/canwell_domainlong_slope2010_5m.tif"
tpi_path = "../data/rasters/canwell_domainlong_tpi2010_5m.tif"
tri_path = "../data/rasters/canwell_domainlong_tri2010_5m.tif" 

fddem_path = "../data/rasters/canwell_domainlong_ddem_2025_2010_5m.tif" # target

# basin outline
basin_path  = "../data/outlines/ncanwell_longbasin_nolap.gpkg"
print("paths set")

paths set


In [2]:
# 0.1 basin mask from outline polygon
import rasterio
import geopandas as gpd
from rasterio.features import geometry_mask

def make_basin_mask(dem_file, basin_file):
    """
    Rasterize a basin outline polygon onto the DEM grid.
    Returns: basin_mask (bool, True inside basin), transform, extent [L,R,B,T].
    """
    with rasterio.open(dem_file) as src:
        dem_crs   = src.crs
        transform = src.transform
        shape     = (src.height, src.width)
        b         = src.bounds
    extent = [b.left, b.right, b.bottom, b.top]

    basin_poly = gpd.read_file(basin_file).to_crs(dem_crs)
    basin_mask = geometry_mask(
        basin_poly.geometry,
        out_shape=shape,
        transform=transform,
        invert=True,
    )
    return basin_mask, transform, extent

In [3]:
# priority-flood + epsilon fill toward the basin outlet (routing DEM only)
# Barnes priority flood algorithm (Barnes, Lehman, Mulla, 2014)
import heapq
import numpy as np

def priority_flood_fill(dem, valid, basin_mask, eps=1e-3):
    H, W = dem.shape
    filled = dem.astype(np.float64).copy()
    done = np.zeros((H, W), bool)
    pq = []
    for r, c in zip(*np.where(basin_mask & valid)):   # basin = outlets
        done[r, c] = True
        heapq.heappush(pq, (filled[r, c], r, c))
    nb = [(-1,0),(1,0),(0,1),(0,-1),(-1,-1),(-1,1),(1,-1),(1,1)]
    while pq:
        e, r, c = heapq.heappop(pq)
        for dr, dc in nb:
            nr, nc = r+dr, c+dc
            if 0 <= nr < H and 0 <= nc < W and valid[nr, nc] and not done[nr, nc]:
                done[nr, nc] = True
                filled[nr, nc] = max(filled[nr, nc], e + eps)   # just-above downstream
                heapq.heappush(pq, (filled[nr, nc], nr, nc))
    reached = int((done & valid).sum()); total = int(valid.sum())
    print(f"flood reached {reached:,}/{total:,} valid cells")
    return filled

In [4]:
# 0.2 breached D8 directed forest over SLOPE cells only
# using the routing dem to trace paths, orig. dem for elev feature
# --- basin = mushgraph, added later

import numpy as np
import rasterio
import networkx as nx
from scipy.ndimage import distance_transform_edt
import psutil

def diGraph_from_dem(dem_path, aspect_path, eucdist_path, lst_path, ndvi_path, pddem_path, roughness_path,
                     slope_path, tpi_path, tri_path, fddem_path,
                     basin_mask, cell_size=CELL_SIZE, basin_level=-99.0):
    """
    Downhill directed forest over the SLOPE cells of a merged slope+basin DEM. Basin cells
    are floored to basin_level so slope flow drains into the basin; basin cells are NOT made
    into nodes. A slope cell whose steepest descent exits into the basin is a 'pit' (tree root).

    Node attrs (slope only): dem_idx, elev, slope, aspect_sin, aspect_cos, ndvi, lst,
        ddem (target), dist_euclid (m to nearest basin cell), path_to_pit (-1 placeholder),
        is_pit, is_source, is_localSink, is_absSink.   Edge attr: weight = |normalized slope|.
    """

    with rasterio.open(dem_path) as src:
        dem = src.read(1); nodata = src.nodata; H, W = dem.shape

    def read_like(path, name):
        with rasterio.open(path) as s: a = s.read(1)
        if a.shape != (H, W): raise ValueError(f"{name} {a.shape} != DEM {(H,W)}")
        return a    

    # import slope features
    aspect     = read_like(aspect_path, "aspect")
    eucdist_data = read_like(eucdist_path, "eucd_basin")
    ndvi_data  = read_like(ndvi_path, "ndvi")
    lst_data   = read_like(lst_path, "lst")
    pddem_data  = read_like(pddem_path, "pddem") # feature
    roughness_data = read_like(roughness_path, "roughness")
    slope_data = read_like(slope_path, "slope")
    tpi_data   = read_like(tpi_path, "tpi")
    tri_data   = read_like(tri_path, "tri")
    
    # import target
    fddem_data  = read_like(fddem_path, "fddem") # target
    
    # calculate aspect sin, cos
    # aspect2010 has 580 invalid (flat) cells inside domain
    # np.sin/cos would assign garbage value, so remove to 0.0
    asp = np.deg2rad(aspect.astype(np.float64))
    flat = (aspect == -9999)
    asp_sin = np.where(flat, 0, np.sin(asp)).astype(np.float32)
    asp_cos = np.where(flat, 0, np.cos(asp)).astype(np.float32)
    
    # determine valid cells
    # VERIFY that dem nodata is NOT NaN
    # because NaN != NaN inherently, would throw false valid cells
    valid = dem != nodata                    # total real (slope or basin domain) cells
    
    # dem with basin pushed down
    dem_mod = dem.copy(); dem_mod[basin_mask] = basin_level 
    
    # routing dem instead of breaching
    dem_route = priority_flood_fill(dem_mod, valid, basin_mask) 
    
    slope_cell = valid & (~basin_mask)       # slope domain nodes only

    graph = nx.DiGraph()
    sourceList, pitList, absSinkList = [], [], []
    neighbors = [(-1,0),(1,0),(0,1),(0,-1),(-1,-1),(-1,1),(1,-1),(1,1)]

    for r in range(H):
        if r % 100 == 0:
            print(f"Row {r}/{H}, RAM: {psutil.virtual_memory().percent:.1f}%")
        for c in range(W):
            if not slope_cell[r, c]:
                continue # skip nodata and basin cells
            idx = r * W + c # single count across raster
            graph.add_node(idx, dem_idx=[r, c],
                           elev=float(dem[r, c]), slope=float(slope_data[r, c]),
                           aspect_sin=float(asp_sin[r, c]), aspect_cos=float(asp_cos[r, c]),
                           eucdist_basin=float(eucdist_data[r, c]),
                           ndvi=float(ndvi_data[r, c]), lst=float(lst_data[r, c]),
                           pddem=float(pddem_data[r, c]), roughness=float(roughness_data[r, c]),
                           tpi=float(tpi_data[r, c]), tri=float(tri_data[r, c]), 
                           path_dist_pit=-1,                         # filled after trees built
                           fddem=float(fddem_data[r, c]),           # target
                           is_pit=False, is_source=False,
                           is_absSink=False)

            # for a given node:
            minSlope, maxSlope = 0.0, 0.0
            next_node, next_is_basin = idx, False
            for dr, dc in neighbors:
                nr, nc = r + dr, c + dc
                if not (0 <= nr < H and 0 <= nc < W): continue # valid check
                if not valid[nr, nc]: continue # never route into off-domain nodata

                s = (dem_route[nr, nc] - dem_route[r, c]) / (np.hypot(dr, dc) * cell_size)
                if s < minSlope: # steeper downslope->new next node; check for basin termination
                    minSlope = s; next_node = nr * W + nc; next_is_basin=bool(basin_mask[nr, nc])
                if s > maxSlope:
                    maxSlope = s

            # assign node type features
            if minSlope < 0.0: # has outlet
                if next_is_basin:              # steepest descent exits to basin
                    graph.nodes[idx]["is_pit"] = True
                    pitList.append(idx)
                else:
                    graph.add_edge(idx, next_node, weight=abs(minSlope))
                    if minSlope < 0 and maxSlope == 0: # source: outflow, no inflow
                        graph.nodes[idx]["is_source"] = True
                        sourceList.append(idx)
            else: # no (breached) outlet in any direction, no basin
                graph.nodes[idx]["is_absSink"] = True
                absSinkList.append(idx)
                
    print(f"nodes: {graph.number_of_nodes():,}  edges: {graph.number_of_edges():,}")
    print(f"sources {len(sourceList):,} | pits {len(pitList):,}"
          f"| abs sinks {len(absSinkList):,}")                
                    
    return graph, sourceList, pitList, absSinkList, dem, dem_mod        

In [5]:
# build basin mask to use in graph construction
# transform and extent for later use
basin_mask, transform, extent = make_basin_mask(dem_path, basin_path)

In [6]:
# retrieve the directed dem graph and the lists of basin pit and other type nodes
sledGraph, sourceList, basinPitList, absSinkList, dem_data, dem_mod = diGraph_from_dem(
    dem_path, aspect_path, eucdist_path, lst_path, ndvi_path, pddem_path, roughness_path,
    slope_path, tpi_path, tri_path, fddem_path,
    basin_mask, cell_size=CELL_SIZE, basin_level=-99.0)

flood reached 1,723,466/1,723,466 valid cells
Row 0/1545, RAM: 22.3%
Row 100/1545, RAM: 22.2%
Row 200/1545, RAM: 22.3%
Row 300/1545, RAM: 22.4%
Row 400/1545, RAM: 22.6%
Row 500/1545, RAM: 22.8%
Row 600/1545, RAM: 23.1%
Row 700/1545, RAM: 23.4%
Row 800/1545, RAM: 23.7%
Row 900/1545, RAM: 24.0%
Row 1000/1545, RAM: 24.4%
Row 1100/1545, RAM: 24.6%
Row 1200/1545, RAM: 24.8%
Row 1300/1545, RAM: 24.9%
Row 1400/1545, RAM: 24.9%
Row 1500/1545, RAM: 24.9%
nodes: 887,825  edges: 884,105
sources 587 | pits 3,720| abs sinks 0


In [7]:
# 2.1 fill path_dist_pit for all nodes with multisource BFS
from collections import deque
def assign_path_dist_pit(graph, pit_list):
    dq = deque()
    for p in pit_list:
        graph.nodes[p]["path_dist_pit"] = 0
        dq.append(p)
    while dq:
        cur = dq.popleft(); d = graph.nodes[cur]["path_dist_pit"]
        for pred in graph.predecessors(cur):
            if graph.nodes[pred]["path_dist_pit"] == -1:
                graph.nodes[pred]["path_dist_pit"] = d + 1
                dq.append(pred)

    unreached = sum(1 for n in graph if graph.nodes[n]["path_dist_pit"] == -1)
    print(f"path_dist_pit filled | unreached nodes: {unreached:,}")
    return unreached

In [8]:
# assign path_dist_pit feature values
unreached = assign_path_dist_pit(sledGraph, basinPitList)

path_dist_pit filled | unreached nodes: 0


In [11]:
# checkpoint 1:
# checkpoint 1
save_graph = {
    "sledgraph_routebreach2010": sledGraph,    # slope-only directed forest, all node/edge attrs
    "dem_data":    dem_data,               # original elevation (plotting)
    "dem_mod":     dem_mod,                # basin floored to basin_level (routing reference)
    "basinPitList":     basinPitList,                # slope-side toe pits (tree roots)
    "sourceList":  sourceList,
    "absSinkList":   absSinkList,          # empty this run, kept for schema consistency
    "basin_mask":  basin_mask,
    "transform":   transform,
    "extent":      extent,
}

In [12]:
import pickle
with open('../data/constructs/CGTN_sledgraph_routebreach2010.pkl','wb') as f:
    pickle.dump(save_graph,f)
print("saved!")

saved!


In [13]:
# 2.2 -- trace one tree upstream of a pit (for plotting),
#       with each node's edge-count (path) distance from pit (root)

def trace_tree_from_pit(graph, pit_node):
    """
    All nodes flowing into pit_node (BFS over predecessors).
    (BFS -- breadth-first search)
    Returns (tree_subgraph, depth), where depth[node] = edge count to the pit.
    Terminates at one out-edge per node; depth dic guards from cycles
    """

    depth = {pit_node: 0}
    order = [pit_node]; i = 0
    while i < len(order):
        cur = order[i]; i += 1
        for p in graph.predecessors(cur):
            if p not in depth:
                depth[p] = depth[cur] + 1
                order.append(p)
    return graph.subgraph(depth.keys()).copy(), depth

In [14]:
# 2.3 keep drainage trees larger than a threshold (50)
basinPit_trees = {}
threshold = 50
for pit_idx in basinPitList:
    tree, depth = trace_tree_from_pit(sledGraph, pit_idx)
    if tree.number_of_nodes() > threshold:
        basinPit_trees[pit_idx] = tree
print(f"basinPit trees size > {threshold}: {len(basinPit_trees)}")

basinPit trees size > 50: 522


In [16]:
threshold_trees = basinPit_trees.keys()
threshold_tree_sizes = [basinPit_trees[tree].number_of_nodes() for tree in threshold_trees]
threshold_tree_coverage = np.sum(threshold_tree_sizes)
print("threshold_tree_coverage:", threshold_tree_coverage, " nodes ", 
      "| ", (tree_node_coverage/sledGraph.number_of_nodes())*100, "%")

threshold_tree_coverage: 851862  nodes  |  95.94931433559542 %


In [33]:
import numpy as np
from scipy.spatial import cKDTree

N_s = threshold_tree_coverage                      # 851,862

# ---- node -> torch idx, and each node's pit ----
# for ALL valid nodes in threshold trees
node2idx, node_pit_key = {}, {}
for pit_key, tree in basinPit_trees.items():       # SAME order the flatten will use
    for n in tree.nodes():
        if n not in node2idx:
            node2idx[n] = len(node2idx) # assign torch order index
            node_pit_key[n] = pit_key   # map each node to its pit
assert len(node2idx) == N_s, (len(node2idx), N_s)

# now map the torch-ordered indices to torch-ordered pits
node_pit_idx = np.empty(N_s, dtype=np.int64)
for n, i in node2idx.items():
    node_pit_idx[i] = node2idx[node_pit_key[n]]    # torch idx of this node's pit

# ---- pits in torch-index space + grid coords ----
pit_keys  = list(basinPit_trees.keys())
pit_torch = np.array([node2idx[k] for k in pit_keys], dtype=np.int64)
pit_rc    = np.array([basinPit_trees[k].nodes[k]["dem_idx"] for k in pit_keys], dtype=np.float64)  # (P,2) [r,c]

# guard: each pit maps to itself (index with torch idx, not node keys)
assert (node_pit_idx[pit_torch] == pit_torch).all(), "pit self-map broken — ordering mismatch"
print("node→pit map OK | non-pit nodes:", int((np.arange(N_s) != node_pit_idx).sum()))
print("check: threshold_coverage - number of pits: ", threshold_tree_coverage - len(basinPit_trees))

# ---- node -> pit edges (drop pit self-loops) ----
sids = np.arange(N_s) # slope_ids # note: node2idx just counts 0 to N_s
m = sids != node_pit_idx # keep only those that are not pits
e_pitskip = np.stack([sids[m], node_pit_idx[m]])  # (2, N_s - P)
print(f"pits {P} | node→pit edges {e_pitskip.shape[1]:,}")

node→pit map OK | non-pit nodes: 851340
check: threshold_coverage - number of pits:  851340
pits 522 | node→pit edges 851,340


# ---- pit -> 2 nearest pits (internal connectivity) ----
P   = len(pit_keys)
kdt = cKDTree(pit_rc)
_, nn = kdt.query(pit_rc, k=3)                              # col0 = self, cols 1,2 = nearest two
e_pitpit = np.stack([np.repeat(pit_torch, 2),               # twice each pit, one for each of two closest
                     pit_torch[nn[:, 1:3].reshape(-1)]])    # (2, 2P)

print(f"pits {P} | node→pit edges {e_pitskip.shape[1]:,} | pit→pit edges {e_pitpit.shape[1]:,}")

# verify that pit-pit edges form a connected graph

from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

rows = np.repeat(np.arange(P), 2)
cols = nn[:, 1:3].reshape(-1)
A = coo_matrix((np.ones(rows.size), (rows, cols)), shape=(P, P))
n_comp, labels = connected_components(A, directed=False)   # directed=False -> undirected
print(f"pit–pit components: {n_comp}")
if n_comp > 1:
    print("component sizes:", sorted(np.bincount(labels), reverse=True))

In [34]:
# 72 components using cKDTree, build with minimum spanning tree instead

from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.spatial.distance import pdist, squareform

mst = minimum_spanning_tree(squareform(pdist(pit_rc))).tocoo()   # P-1 edges, local idx
e_pitpit = np.concatenate(
    [e_pitpit, np.stack([pit_torch[mst.row], pit_torch[mst.col]])], axis=1)

# re-verify
A2 = A + coo_matrix((np.ones(mst.nnz), (mst.row, mst.col)), shape=(P, P))
print("components after MST union:", connected_components(A2, directed=False)[0])

components after MST union: 1


In [32]:
# flatten to .npz for torch training
import numpy as np

FEATS = ["elev","slope","aspect_sin","aspect_cos","roughness","tri","tpi",
         "ndvi","lst","eucdist_basin","pddem","path_dist_pit"]        # 12 features
TARGET = "fddem"                                                    # 2025-2010

X = np.empty((N_s, len(FEATS)), dtype=np.float32)
y = np.empty(N_s, dtype=np.float32)
node_rc = np.empty((N_s, 2), dtype=np.int32)

# build torch edges from threshold trees
# and assign feature values to torch node indices
src, dst = [], []
for pit_key, tree in basinPit_trees.items():
    for n, a in tree.nodes(data=True):
        i = node2idx[n]
        X[i] = [a[f] for f in FEATS] # feat values at each node
        y[i] = a[TARGET]             # target value at each node
        node_rc[i] = a["dem_idx"]    # each node's raster row, column
    for u, v in tree.edges():                       # directed downslope
        # relabel edges with torch index endpoints
        src.append(node2idx[u]); dst.append(node2idx[v]) 
e_tree = np.array([src, dst], dtype=np.int64)
print("torch tree edges: ", e_tree.shape[1])

torch tree edges:  851340


In [36]:
# sentinel/NaN sweep (the -3.4e38 trap) -> NaN, which HistGBT handles natively and zscore+nan_to_num handles for the GNN
X[X <= -1e30] = np.nan
y[y <= -1e30] = np.nan

print("feature sentinel/NaN cells:", int(np.isnan(X).sum()), "| y NaN:", int(np.isnan(y).sum()))

np.savez("../data/constructs/sledG_torch_pitedge_pitpit_2010.npz",
         X=X, y=y, feat_names=np.array(FEATS), node_rc=node_rc,
         node_pit_idx=node_pit_idx, pit_torch=pit_torch, pit_rc=pit_rc,
         e_tree=e_tree, e_pitskip=e_pitskip, e_pitpit=e_pitpit)
print(f"nodes {N_s:,} | feats {len(FEATS)} | tree edges {e_tree.shape[1]:,} "
      f"| node→pit {e_pitskip.shape[1]:,} | pit→pit {e_pitpit.shape[1]:,}")

feature sentinel/NaN cells: 0 | y NaN: 6
nodes 851,862 | feats 12 | tree edges 851,340 | node→pit 851,340 | pit→pit 2,086


In [ ]:
# next, connected mushgraph version

In [37]:
# connect drainage trees to basin (mush)graph
import numpy as np
from scipy.spatial import cKDTree
import rasterio

# basin mushgraph
bg = np.load("../data/constructs/basin_mushgraph2010.npz", allow_pickle=True)

blevel, bx, by = bg["level"], bg["x"], bg["y"]
feat_names = list(bg["feat_names"])
belev = bg["X"][:, feat_names.index("elev")]
all_idx = np.arange(len(blevel)) # assign node indices for all basin nodes
L0, L2 = all_idx[blevel == 0], all_idx[blevel == 2] # subselect levels 0 and 2

# KDTree -- nearest-neighbor lookups in K-dimensional space
tree0 = cKDTree(np.c_[bx[L0], by[L0]]) # build lookup trees for the x,y coord

pit_nodes = threshold_trees       # 252 pit roots (as sledGraph node ids)
tree_sizes = threshold_tree_sizes # node counts of the trees
DEM_FILE   = dem_path             # domain dem the full sledGraph was built on
P = len(pit_nodes)

with rasterio.open(DEM_FILE) as src:
    T = src.transform

pit_rc = np.array([sledGraph.nodes[n]["dem_idx"] for n in pit_nodes])
pit_elev   = np.array([sledGraph.nodes[n]["elev"]    for n in pit_nodes])

#pit_xy     = np.array([idx_xy(i) for i in pit_demidx]) # coordinates for all pit nodes

rows, cols = pit_rc[:, 0], pit_rc[:, 1]
pit_xy = np.c_[T.c + (cols + 0.5) *T.a,   # x = origin_x + (col + 1/2)*pixel
                T.f + (rows + 0.5) * T.e]  # y = origin_y + (row + 1/2)*pixel(neg)

# L0: each pit connects to nearest L0 node
d0, j0 = tree0.query(pit_xy) # distances, indices <-- query
b0 = L0[j0] # level-0 basin nodes closest to the queried pit nodes

# L2: each pit connects to all L2 nodes
pit_L2 = np.repeat(np.arange(P), len(L2))
b2_all = np.tile(L2, P)


def edge_attr(pit_i, b):
    dx = bx[b] - pit_xy[pit_i,0] # distance in x, y coordinates
    dy = by[b] - pit_xy[pit_i,1]
    d = np.hypot(dx, dy)
    u = np.divide(1.0, d, out=np.zeros_like(d), where=(d>0)) # 1/d where d>0; 0 otherwise
    return np.c_[d, dx*u, dy*u, belev[b] - pit_elev[pit_i]] # 1D arrays into columns of 2D array

link_pit   = np.r_[np.arange(P), pit_L2]
link_basin = np.r_[b0,           b2_all]
link_kind  = np.r_[np.zeros(P, int), np.ones(len(pit_L2),int)]
link_attr  = np.vstack([edge_attr(np.arange(P), b0),
                        edge_attr(pit_L2,  b2_all)])

print(f"{P} pits wired | L0 nn: {d0.mean():.0f}±{d0.std():.0f} m (max {d0.max():.0f}) ")

522 pits wired | L0 nn: 7±3 m (max 11) 


In [38]:
np.savez("../data/constructs/pit_basin_links2010.npz",
         pit_dem_idx=pit_rc, pit_xy=pit_xy, pit_elev=pit_elev, tree_size=tree_sizes,
         link_pit=link_pit, link_basin=link_basin, link_kind=link_kind, link_attr=link_attr,
         attr_names=np.array(["dist_m","ux","uy","dz"]))

print("pit_basin_links2010.npz saved")

pit_basin_links2010.npz saved
